In [127]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as sts
from tqdm.notebook import tqdm

In [128]:
def convertor_disc(strings):
    strings = str(strings).replace('<span class="css-1l9tp44 e162wx9x0" data-ftid="bull_description-item">', '').replace('<!-- -->,</span>','')
    return strings

In [129]:
def convertor_loc(strings):
    strings = str(strings).replace('<span class="css-1488ad e162wx9x0" data-ftid="bull_location">', '').replace('</span>','')
    return strings

In [130]:
def convertor_title(strings):
    strings = str(strings).replace('<h3 class="css-16kqa8y efwtv890">', '').replace('</h3>','')
    return strings

In [131]:
def convertor_price(strings):
    strings = str(strings).replace('<span class="css-46itwz e162wx9x0"><span data-ftid="bull_price">', '').replace('\xa0','').replace('<!-- --></span>',' ').replace('</span>','')
    return strings

In [151]:
data_char, data_city, data_title, data_price = [], [], [], []
min_pr = 0
for s in tqdm(range(8)):
    for k in tqdm(range(1,101)):

        url = 'https://auto.drom.ru/moto/motorcycle/all/page' + str(k) + '/?order=price&minprice=' + str(min_pr+1) + '&unsold=1'

        response = requests.get(url)
        tree = BeautifulSoup(response.content, 'html.parser') # строим дерево

        char = tree.find_all('span', {'class' : "css-1l9tp44 e162wx9x0"})

        char_engine_cap, char_mileage, char_tact, char_transmission, char_fuel_supply_type, char_oil = '-', '-', '-', '-', '-', '-'
        for c in range(len(char)):
            obj = convertor_disc(char[c])
            if '</span>' not in obj:
                if 'куб. см' in obj:
                    char_engine_cap = obj
                elif 'км' in obj:
                    char_mileage = obj
                elif 'тактный' in obj:
                    char_tact = obj
                elif 'механика' in obj or 'автомат' in obj:
                    char_transmission = obj
                elif 'карбюратор' in obj or 'инжектор' in obj:
                    char_fuel_supply_type = obj
                elif 'бензин' in obj or 'газ' in obj or 'дизель' in obj or 'электро' in obj:
                    char_oil = obj
            else:
                obj = obj.replace('</span>','')
                if 'куб. см' in obj:
                    char_engine_cap = obj
                elif 'км' in obj:
                    char_mileage = obj
                elif 'тактный' in obj:
                    char_tact = obj
                elif 'механика' in obj or 'автомат' in obj:
                    char_transmission = obj
                elif 'карбюратор' in obj or 'инжектор' in obj:
                    char_fuel_supply_type = obj
                elif 'бензин' in obj or 'газ' in obj or 'дизель' in obj or 'электро' in obj:
                    char_oil = obj
                data_char.append({'Пробег': char_mileage,
                                  'Объём двигателя': char_engine_cap,
                                  'Тип топлива': char_oil,
                                  'Тип подачи топлива': char_fuel_supply_type,
                                  'Тактность': char_tact,
                                  'Коробка передач': char_transmission})
                char_engine_cap, char_mileage, char_tact, char_transmission, char_fuel_supply_type, char_oil = '-', '-', '-', '-', '-', '-'

        city = tree.find_all('span', {'class' : "css-1488ad e162wx9x0"})

        for j in range(len(city)):
            obj = convertor_loc(city[j])
            data_city.append({'Город': obj})

        name_year = tree.find_all('h3', {'class' : "css-16kqa8y efwtv890"})    

        for n in range(len(name_year)):
            obj = convertor_title(name_year[n]).split(',')
            data_title.append({'Модель': obj[0],
                               'Год выпуска': obj[1]})

        price = tree.find_all('span', {'class' : "css-46itwz e162wx9x0"})

        for p in range(len(price)):
            obj = convertor_price(price[p])
            data_price.append({'Стоимость': obj})
            if p == len(price)-1 and k == 100:
                min_pr = int(obj[:-2])
    print(min_pr)

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

110000


  0%|          | 0/100 [00:00<?, ?it/s]

175000


...

677500


  0%|          | 0/100 [00:00<?, ?it/s]

989000


  0%|          | 0/100 [00:00<?, ?it/s]

989000


In [152]:
data_char_df = pd.DataFrame(data_char)
data_city_df = pd.DataFrame(data_city)
data_title_df = pd.DataFrame(data_title)
data_price_df = pd.DataFrame(data_price)

In [153]:
data = data_price_df.join(data_title_df).join(data_city_df).join(data_char_df)
data

,Стоимость,Модель,Год выпуска,Город,Пробег,Объём двигателя,Тип топлива,Тип подачи топлива,Тактность,Коробка передач
0,86 ₽,Cobra Crossfire,2013,Уфа,10 км,150 куб. см.,бензин,карбюратор,4-тактный,механика
1,6000 ₽,Урал ИМЗ 8.103-10,1991,Пограничный,10 км,499 куб. см.,бензин,-,4-тактный,-
2,6500 ₽,Днепр 16М,1985,Барабинск,-,450 куб. см.,-,карбюратор,-,механика
3,8000 ₽,Восход Восход,1981,Кызыл,-,175 куб. см.,-,-,-,-
4,9999 ₽,Honda CBR 919RR,1999,Спасск-Дальний,50 000 км,919 куб. см.,бензин,-,4-тактный,-
...,...,...,...,...,...,...,...,...,...,...
15458,990000 ₽,Yamaha FJR 1300AS,2013,Томск,33 000 км,1300 куб. см.,бензин,инжектор,4-тактный,автомат
15459,990000 ₽,Honda CMX 1100,2022,Москва,2 150 км,1100 куб. см.,бензин,-,4-тактный,-
15460,990000 ₽,Kawasaki Z 900,2021,Геленджик,24 000 км,950 куб. см.,бензин,-,4-тактный,-
15461,990000 ₽,Kawasaki Z 650,2022,Москва,10 500 км,650 куб. см.,бензин,инжектор,4-тактный,механика


In [169]:
data = data[:15458]

In [170]:
data.to_csv('drom.csv', encoding='utf-8')